**TASK 1 – Credit Risk Modelling (GermanCredit.csv)**

In [ ]:
import pandas as pd

data = pd.read_csv('GermanCredit.csv')
data.head()
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Unnamed: 0            1000 non-null   int64  
 1   checking_balance      606 non-null    float64
 2   months_loan_duration  1000 non-null   int64  
 3   credit_history        1000 non-null   object 
 4   purpose               1000 non-null   object 
 5   amount                1000 non-null   int64  
 6   savings_balance       817 non-null    float64
 7   employment_length     938 non-null    object 
 8   installment_rate      1000 non-null   int64  
 9   personal_status       690 non-null    object 
 10  other_debtors         1000 non-null   object 
 11  residence_history     870 non-null    object 
 12  property              1000 non-null   object 
 13  age                   1000 non-null   int64  
 14  installment_plan      1000 non-null   object 
 15  housing               

If any numeric columns have missing values → fill with median.
If categorical → fill with 'MISSING'

In [ ]:
for col in data.select_dtypes(include='number'):
    data[col] = data[col].fillna(data[col].median())

for col in data.select_dtypes(include='object'):
    data[col] = data[col].fillna('MISSING')

Create new useful features
(These features help the model understand loan-to-savings balance and loan size relative to median.)

In [ ]:
data['amount_to_savings_ratio'] = data['amount'] / (data['savings_balance'] + 1)
data['large_loan_flag'] = (data['amount'] > data['amount'].median()).astype(int)

In [ ]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 25 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   application_id           1000 non-null   int64  
 1   checking_balance         1000 non-null   float64
 2   months_loan_duration     1000 non-null   int64  
 3   credit_history           1000 non-null   object 
 4   purpose                  1000 non-null   object 
 5   amount                   1000 non-null   int64  
 6   savings_balance          1000 non-null   float64
 7   employment_length        1000 non-null   object 
 8   installment_rate         1000 non-null   int64  
 9   personal_status          1000 non-null   object 
 10  other_debtors            1000 non-null   object 
 11  residence_history        1000 non-null   object 
 12  property                 1000 non-null   object 
 13  age                      1000 non-null   int64  
 14  installment_plan         

**Spliting the Data:**
Separate the features (X) and target (y):

In [ ]:
X = data.drop(columns=['default'])
y = data['default']


Then split into training and testing sets:

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)


**Preprocess and Train Model:**

Encode categorical variables and scale numeric ones.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier

numeric_cols = X.select_dtypes(include='number').columns
cat_cols = X.select_dtypes(include='object').columns

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), numeric_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

model = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(n_estimators=150, random_state=42))
])

model.fit(X_train, y_train)


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  Index(['application_id', 'checking_balance', 'months_loan_duration', 'amount',
       'savings_balance', 'installment_rate', 'age', 'existing_credits',
       'dependents', 'telephone', 'amount_to_savings_ratio',
       'large_loan_flag'],
      dtype='object')),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  Index(['credit_history', 'purpose', 'employment_length', 'personal_status',
       'other_debtors', 'residence_history', 'property', 'installment_plan',
       'housing', 'foreign_worker', 'job', 'gender'],
      dtype='object'))])),
                ('classifier',
                 RandomForestClassifier(n_estimators=150, random_state=42))])

**Evaluate the Model**

Check accuracy and performance:

In [ ]:
from sklearn.metrics import classification_report, roc_auc_score

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

print(classification_report(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_proba))


              precision    recall  f1-score   support

           0       0.75      0.95      0.84       175
           1       0.70      0.25      0.37        75

    accuracy                           0.74       250
   macro avg       0.73      0.60      0.61       250
weighted avg       0.74      0.74      0.70       250

ROC AUC: 0.7804571428571428


**Save and Reuse Model:**

In [ ]:
import joblib
joblib.dump(model, 'credit_risk_model.pkl')


['credit_risk_model.pkl']

In [ ]:
loaded_model = joblib.load('credit_risk_model.pkl')
# new_customer_data needs to be defined with the same columns as the training data
# For demonstration, we will use X_test as sample new_customer_data
new_customer_data = X_test
new_scores = loaded_model.predict(new_customer_data)
display(new_scores)

array([1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0,
       0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0])

**Task 2 – Credit Bureau Feature Engineering (using the uploaded JSON)**

In [ ]:
import json

with open("Credit_bureau_sample_data.json", "r") as f:
    bureau_data = json.load(f)

print(len(bureau_data))
print(bureau_data[0].keys())   # should show 'application_id' and 'data'

3
dict_keys(['application_id', 'data'])


**Inspect a Sample Credit Report**

In [ ]:
sample_report = bureau_data[0]["data"]
pd.json_normalize(sample_report).head()


,consumerfullcredit.subjectlist.reference,consumerfullcredit.subjectlist.consumerid,consumerfullcredit.subjectlist.searchoutput,consumerfullcredit.accountrating.noofotheraccountsbad,consumerfullcredit.accountrating.noofotheraccountsgood,consumerfullcredit.accountrating.noofretailaccountsbad,consumerfullcredit.accountrating.noofretailaccountsgood,consumerfullcredit.accountrating.nooftelecomaccountsbad,consumerfullcredit.accountrating.noofautoloanaccountsbad,consumerfullcredit.accountrating.noofautoloanccountsgood,...,consumerfullcredit.accountmonthlypaymenthistoryheader.mh18,consumerfullcredit.accountmonthlypaymenthistoryheader.mh19,consumerfullcredit.accountmonthlypaymenthistoryheader.mh20,consumerfullcredit.accountmonthlypaymenthistoryheader.mh21,consumerfullcredit.accountmonthlypaymenthistoryheader.mh22,consumerfullcredit.accountmonthlypaymenthistoryheader.mh23,consumerfullcredit.accountmonthlypaymenthistoryheader.mh24,consumerfullcredit.accountmonthlypaymenthistoryheader.company,consumerfullcredit.accountmonthlypaymenthistoryheader.tablename,consumerfullcredit.accountmonthlypaymenthistoryheader.displaytext
0,12876566,17628566,XXX,0,3,0,2,0,0,0,...,2019\nMAR,2019\nFEB,2019\nJAN,2018\nDEC,2018\nNOV,2018\nOCT,2018\nSEP,Company,Consumer24MonthlyPaymentHeader,Consumer 24 Monthly Payment Header


**Extract Features**

In [ ]:
def extract_credit_features(credit_reports):
    features = []

    for report in credit_reports:
        app_id = report.get('application_id')
        data = report.get('data', {})

        # Example sections you might find
        accounts = data.get('accounts', [])
        inquiries = data.get('inquiries', [])
        collections = data.get('collections', [])

        # --- Basic statistics ---
        num_accounts = len(accounts)
        num_inquiries = len(inquiries)
        num_collections = len(collections)

        # --- Compute totals / aggregates ---
        total_balance = sum(acc.get('balance', 0) for acc in accounts if isinstance(acc, dict))
        total_credit_limit = sum(acc.get('credit_limit', 0) for acc in accounts if isinstance(acc, dict))
        avg_utilization = (total_balance / total_credit_limit) if total_credit_limit > 0 else 0
        num_past_due = sum(1 for acc in accounts if acc.get('status', '').lower() in ['past_due','delinquent'])

        # --- Combine features ---
        features.append({
            'application_id': app_id,
            'num_accounts': num_accounts,
            'num_inquiries': num_inquiries,
            'num_collections': num_collections,
            'total_balance': total_balance,
            'total_credit_limit': total_credit_limit,
            'avg_utilization': round(avg_utilization, 2),
            'num_past_due': num_past_due
        })

    return pd.DataFrame(features)

**Run the code and show me the Results**

In [ ]:
features_df = extract_credit_features(bureau_data)
features_df.head()

,application_id,num_accounts,num_inquiries,num_collections,total_balance,total_credit_limit,avg_utilization,num_past_due
0,97,0,0,0,0,0,0,0
1,9714953,0,0,0,0,0,0,0
2,9714978,0,0,0,0,0,0,0


In [ ]:
from google.colab import sheets
sheet = sheets.InteractiveSheet(df=features_df)

NameError: name 'features_df' is not defined

**Explain Each Feature (Markdown)**

**Feature explanations**
- `num_accounts`: The total number of active or past accounts; more accounts can mean higher exposure.
- `num_inquiries`: Number of recent credit inquiries; many inquiries may indicate financial pressure.
- `total_balance`: Total outstanding balance; higher amounts increase default risk.
- `avg_utilization`: Credit usage rate; higher utilization suggests riskier behavior.
- `num_past_due`: Number of overdue accounts; strongly predictive of default likelihood.


**Join with Main Dataset**

If both datasets have application_id as a key, you can merge them:

In [ ]:
data = data.rename(columns={'Unnamed: 0': 'application_id'})
final_df = pd.merge(data, features_df, on='application_id', how='left')
final_df.head()

,application_id,checking_balance,months_loan_duration,credit_history,purpose,amount,savings_balance,employment_length,installment_rate,personal_status,...,gender,amount_to_savings_ratio,large_loan_flag,num_accounts,num_inquiries,num_collections,total_balance,total_credit_limit,avg_utilization,num_past_due
0,0,-43.0,6,critical,radio/tv,1169,64.0,13 years,4,single,...,male,17.984615,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,75.0,48,repaid,radio/tv,5951,89.0,2 years,2,MISSING,...,female,66.122222,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2,24.0,12,critical,education,2096,24.0,5 years,2,single,...,male,83.840000,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,3,-32.0,42,repaid,furniture,7882,9.0,5 years,2,single,...,male,788.200000,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,4,-23.0,24,delayed,car (new),4870,43.0,3 years,3,single,...,male,110.681818,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Task 3 – Bike Data Analysis**

In [ ]:
# Load the dataset
df = pd.read_csv("/content/BikerDatav2.csv")
print(df.shape)
df.head()

(5619, 9)


,trip_id,subscriber_type,bikeid,start_time,start_station_id,start_station_name,end_station_id,end_station_name,duration_minutes
0,23298154,Local365,19775,2020-12-10 14:52:24 UTC,2552,3rd/West,2552,3rd/West,41
1,23321039,Pay-as-you-ride,332,2020-12-18 15:43:07 UTC,2552,3rd/West,2552,3rd/West,74
2,23326829,Local365,19171,2020-12-20 15:44:21 UTC,2501,5th/Bowie,2552,3rd/West,62
3,24786257,Local365,19646,2021-08-08 17:31:07 UTC,3390,6th/Brazos,2552,3rd/West,63
4,24743726,Local365,17438,2021-08-03 14:34:41 UTC,4047,8th/Lavaca,2552,3rd/West,6


**Step 2 – Clean the data**

Remove “Missing” / “Stolen” stations and trips where start = end

In [ ]:
print(f"Initial df shape: {df.shape}")
df['start_time'] = pd.to_datetime(df['start_time'])
print(f"Shape after converting start_time to datetime: {df.shape}")
df = df[~df['end_station_name'].isin(['Missing','Stolen'])]
print(f"Shape after removing 'Missing'/'Stolen' end stations: {df.shape}")
df = df[df['start_station_name'] != df['end_station_name']]
print(f"Shape after removing trips where start == end: {df.shape}")
df.info()

Initial df shape: (2676, 9)
Shape after converting start_time to datetime: (2676, 9)
Shape after removing 'Missing'/'Stolen' end stations: (2676, 9)
Shape after removing trips where start == end: (2676, 9)
<class 'pandas.core.frame.DataFrame'>
Index: 2676 entries, 2 to 5615
Data columns (total 9 columns):
 #   Column              Non-Null Count  Dtype              
---  ------              --------------  -----              
 0   trip_id             2676 non-null   int64              
 1   subscriber_type     2673 non-null   object             
 2   bikeid              2676 non-null   object             
 3   start_time          2676 non-null   datetime64[ns, UTC]
 4   start_station_id    2676 non-null   int64              
 5   start_station_name  2676 non-null   object             
 6   end_station_id      2676 non-null   int64              
 7   end_station_name    2676 non-null   object             
 8   duration_minutes    2676 non-null   int64              
dtypes: datetime64[ns,

**Step 3 – Find the day with the longest average trip**

In [ ]:
df['day_of_week'] = df['start_time'].dt.day_name()
day_avg = df.groupby('day_of_week')['duration_minutes'].mean().sort_values(ascending=False)

print("Average duration by day of week:")
print(day_avg)
print("\nDay with longest average trip:", day_avg.idxmax(), "-", round(day_avg.max(),2), "minutes")

Average duration by day of week:
day_of_week
Sunday       79.914153
Saturday     58.228632
Tuesday      50.755418
Thursday     42.449198
Monday       34.768546
Friday       31.529691
Wednesday    26.319876
Name: duration_minutes, dtype: float64

Day with longest average trip: Sunday - 79.91 minutes


**Step 4 – Find the month / year with the most trips**

In [ ]:
df['year'] = df['start_time'].dt.year
df['month'] = df['start_time'].dt.month

month_counts = df.groupby(['year','month']).size().reset_index(name='trip_count')
month_counts = month_counts.sort_values('trip_count', ascending=False)

print("Month/year with most trips:")
print(month_counts.head(1))


Month/year with most trips:
    year  month  trip_count
10  2020     11         248


**Step 5 – Find the longest and shortest trips**

In [ ]:
# Longest trip (if ties, earliest start_time)
longest_trip = df.sort_values(['duration_minutes','start_time'], ascending=[False, True]).head(1)

# Shortest trip
shortest_trip = df.sort_values(['duration_minutes','start_time'], ascending=[True, True]).head(1)

print("Longest trip record:")
display(longest_trip)

print("Shortest trip record:")
display(shortest_trip)


Longest trip record:


,trip_id,subscriber_type,bikeid,start_time,start_station_id,start_station_name,end_station_id,end_station_name,duration_minutes,day_of_week,year,month
1097,21577822,24 Hour Walk Up Pass,2095,2020-02-16 04:37:00+00:00,4054,Rosewood/Chicon,4058,Hollow Creek/Barton Hills,11810,Sunday,2020,2


Shortest trip record:


,trip_id,subscriber_type,bikeid,start_time,start_station_id,start_station_name,end_station_id,end_station_name,duration_minutes,day_of_week,year,month
471,21473408,Pay-as-you-ride,460,2020-01-15 09:14:08+00:00,2822,East 6th/Robert T. Martinez,2544,East 6th/Pedernales,2,Wednesday,2020,1


**Step 6 – Optional: Save the answers to a summary file**

In [ ]:
summary = {
    "Day with longest average trip": day_avg.idxmax(),
    "Avg duration on that day": round(day_avg.max(),2),
    "Top month/year": f"{month_counts.iloc[0]['year']}-{int(month_counts.iloc[0]['month'])}",
    "Trip count that month": int(month_counts.iloc[0]['trip_count']),
    "Longest trip id": int(longest_trip['trip_id'].iloc[0]),
    "Longest trip duration": int(longest_trip['duration_minutes'].iloc[0]),
    "Shortest trip id": int(shortest_trip['trip_id'].iloc[0]),
    "Shortest trip duration": int(shortest_trip['duration_minutes'].iloc[0])
}

summary_df = pd.DataFrame([summary])
summary_df.to_csv("bike_analysis_summary.csv", index=False)
summary_df


,Day with longest average trip,Avg duration on that day,Top month/year,Trip count that month,Longest trip id,Longest trip duration,Shortest trip id,Shortest trip duration
0,Sunday,79.91,2020-11,248,21577822,11810,21473408,2


from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['index']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'index'}, axis=1)
              .sort_values('index', ascending=True))
  xs = counted['index']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_0.sort_values('index', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('index')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['Avg duration on that day']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'Avg duration on that day'}, axis=1)
              .sort_values('Avg duration on that day', ascending=True))
  xs = counted['Avg duration on that day']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_1.sort_values('Avg duration on that day', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('Avg duration on that day')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['Trip count that month']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'Trip count that month'}, axis=1)
              .sort_values('Trip count that month', ascending=True))
  xs = counted['Trip count that month']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_2.sort_values('Trip count that month', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('Trip count that month')
_ = plt.ylabel('count()')

from matplotlib import pyplot as plt
import seaborn as sns
def _plot_series(series, series_name, series_index=0):
  palette = list(sns.palettes.mpl_palette('Dark2'))
  counted = (series['Longest trip id']
                .value_counts()
              .reset_index(name='counts')
              .rename({'index': 'Longest trip id'}, axis=1)
              .sort_values('Longest trip id', ascending=True))
  xs = counted['Longest trip id']
  ys = counted['counts']
  plt.plot(xs, ys, label=series_name, color=palette[series_index % len(palette)])

fig, ax = plt.subplots(figsize=(10, 5.2), layout='constrained')
df_sorted = _df_3.sort_values('Longest trip id', ascending=True)
_plot_series(df_sorted, '')
sns.despine(fig=fig, ax=ax)
plt.xlabel('Longest trip id')
_ = plt.ylabel('count()')